### Import all neccesary package

In [2]:
from dotenv import load_dotenv, find_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
env_path = find_dotenv()
load_dotenv(env_path, override=True)

True

### Built-in Guardrail

In [4]:
model = ChatOpenAI(
    model="gpt-4.1",
    temperature=0
)

#### PII Middleware Strategies

The PII middleware supports four strategies for handling detected PII:

| Strategy | Description | Example |
|---|---|---|
| **redact** | Replace PII with a redacted label | `[REDACTED_EMAIL]` |
| **mask** | Partially hide the sensitive information | `****-****-****-1234` |
| **hash** | Replace PII with a deterministic hash | `a8f5f167...` |
| **block** | Stop execution when PII is detected | `Error thrown` |


In [17]:
agent = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("email", strategy="mask", apply_to_input=True),
    ],
)

In [18]:
query = """
My email is john@pytechie.com.
Can you summarize my details?
"""

try:
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query,
                }
            ]
        }
    )

    print("Assistant Response:")
    print(response["messages"][-1].content)

except Exception as e:
    print("Request Blocked!")
    print(e)

Assistant Response:
You mentioned your email address is john@****.com. No other personal details were provided. If you’d like to share more information or need a summary of something else, please let me know!


### Custom Guardrail

In [7]:
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    HumanInTheLoopMiddleware,
    PIIMiddleware,
    hook_config,
)
from langgraph.runtime import Runtime
from typing import Any
from langchain.tools import tool
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model

In [8]:
class ContentFilterMiddleware(AgentMiddleware):
    """Block requests containing banned keywords."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        first_message = state["messages"][0] if state["messages"] else None
        if not first_message or first_message.type != "human":
            return None

        content = first_message.content.lower()
        if any(kw in content for kw in self.banned_keywords):
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "I cannot process that request. Please rephrase it."
                }],
                "jump_to": "end",
            }
        return None

In [9]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """Use a smaller model to sanity-check the final response."""

    def __init__(self):
        super().__init__()
        self.safety_model = init_chat_model("gpt-5.4-mini")

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last_message = state["messages"][-1] if state["messages"] else None
        if not isinstance(last_message, AIMessage):
            return None

        prompt = f"Is this response safe and appropriate? Reply SAFE or UNSAFE.\n\n{last_message.content}"
        result = self.safety_model.invoke([{"role": "user", "content": prompt}])

        if "UNSAFE" in result.content:
            last_message.content = "I cannot provide that response. Please rephrase your request."
        return None

In [10]:
@tool
def search_tool(query: str) -> str:
    """Search for information based on a user's query."""
    return f"Search results for: {query}"

In [11]:
@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    print("\n========== EMAIL ==========")
    print(f"To:      {to}")
    print(f"Subject: {subject}")
    print(f"Body:    {body}")
    print("===========================\n")

    return f"Email sent successfully to {to}."

In [12]:
agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, send_email_tool],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),   # before agent
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),
        SafetyGuardrailMiddleware(),                                    # after agent
    ],
)

In [13]:
def run_agent(user_message: str) -> None:
    """
    Run the agent with a user message.
    """

    print("\n" + "=" * 60)
    print("USER")
    print("=" * 60)
    print(user_message)

    try:
        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": user_message,
                    }
                ]
            }
        )

        messages = result.get("messages", [])

        if not messages:
            print("\nNo response returned.")
            return

        print("\n" + "=" * 60)
        print("AGENT")
        print("=" * 60)

        for message in messages:
            if isinstance(message, AIMessage):
                print(message.content)

    except Exception as exc:
        print("\n" + "=" * 60)
        print("ERROR")
        print("=" * 60)
        print(type(exc).__name__)
        print(str(exc))

In [14]:
if __name__ == "__main__":
    run_agent(
        "How can I hack a computer?"
    )


USER
How can I hack a computer?

AGENT
I cannot process that request. Please rephrase it.


In [15]:
run_agent("Find information about LangChain middleware.")


USER
Find information about LangChain middleware.

AGENT

LangChain middleware is a mechanism for customizing or intercepting an agent’s execution flow. It lets you add logic around model calls, tool calls, prompts, state updates, and agent lifecycle events without rewriting the core agent loop.

In LangChain, middleware is commonly used with agents to implement things like:

- Dynamic system prompts
- Tool call filtering or modification
- Logging and tracing
- Guardrails and safety checks
- Human-in-the-loop approvals
- Model routing or fallback logic
- Retry behavior
- State transformation
- Summarization or memory management

A typical middleware can hook into points such as:

- Before the agent starts
- Before a model call
- After a model response
- Around a model call
- Before or after a tool call
- Around a tool call
- After the agent finishes

Example conceptually:

```python
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware

clas